# Naključni gozdovi — A: prikaz (telekom)
**Akademija UI za poslovne aplikacije — Modul 2, Nadzorovano učenje v praksi**

Ta zvezek poganja predavatelj. Vajo naredite v zvezku **`B_nakljucni_gozdovi`**.

Vprašanje: **zakaj je veliko različnih dreves boljših od enega?**

### Ponovitev
- **Eno drevo je nestabilno:** če se podatki malo spremenijo, drevo postavi druga vprašanja.
- **Naključni gozd** je veliko različnih dreves. Napoved gozda je povprečje napovedi dreves.
- **Od kod razlike med drevesi:**
  1. vsako drevo se uči na **naključnem vzorcu vrstic** (z vračanjem),
  2. pri vsakem vprašanju sme izbirati samo med **naključnim delom značilk**.
- **Verjetnost gozda** je povprečje verjetnosti posameznih dreves.
- **Pridobimo:** boljše in bolj stabilne napovedi. **Izgubimo:** enega samega drevesa ne moremo več prebrati.
- **Glavne nastavitve:** `n_estimators` (število dreves), `min_samples_leaf`, `max_features`.

### Nastavitev okolja
Zaženite spodnji dve celici (**Shift + Enter**). Ničesar ne spreminjajte. Koda prve celice je skrita.
Druga celica pokliče skupno pripravo podatkov iz `utils.modeliranje`. Veljavne pripravljene tabele ponovno uporabi.

In [ ]:
#@title Nastavitev okolja (kliknite za prikaz kode)
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/valira-ai/ui-za-poslovne-aplikacije.git"

if "google.colab" in sys.modules:
    KOREN = Path("/content/ui-za-poslovne-aplikacije")
    if not (KOREN / "utils/orodja.py").is_file():
        if KOREN.exists():
            raise FileNotFoundError(f"Mapa {KOREN} že obstaja, vendar v njej ni utils/orodja.py.")
        subprocess.run(["git", "clone", "-q", REPO_URL, str(KOREN)], check=True)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", str(KOREN / "requirements.txt")],
        check=True,
    )
else:
    KOREN = Path.cwd().resolve()
    while not (KOREN / "utils/orodja.py").is_file() and KOREN != KOREN.parent:
        KOREN = KOREN.parent
    if not (KOREN / "utils/orodja.py").is_file():
        raise FileNotFoundError("Zaženite zvezek v prenesenem repozitoriju ali njegovi podmapi (manjka utils/orodja.py).")

if str(KOREN) not in sys.path:
    sys.path.insert(0, str(KOREN))
print("Gradivo je v:", KOREN)


In [ ]:
from utils.modeliranje import poskrbi_za_modeliranje, preberi_shemo, preberi_tabelo
PODATKI = poskrbi_za_modeliranje("telekom")
shema = preberi_shemo(PODATKI / "shema.json")


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from plotnine import *
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

from utils.orodja import nastavi_slog, MODRA, ORANZNA, ZELENA, SIVA, SEME, TELEKOM_STEVILSKE
from utils.orodja import narisi_drevo, vrstica_lestvice
nastavi_slog()

<details>
<summary><b>Kako so pripravljeni podatki? (klikni)</b></summary>

Podatke je pripravila skripta `scripts/pripravi_telekom.py`. Naredi isto kot zvezka predavatelja
`A1_Priprava_Telekom` (mapa `03_Uvod_v_strojno_ucenje/notebooks/01_priprava_podatkov`) in
`A1_Delitev_in_manjkajoce_Telekom` (mapa `03_Uvod_v_strojno_ucenje/notebooks/02_preverjanje_modela`).
Tam so vse odločitve z razlogi. Na kratko:

1. Ena vrstica = ena stranka (100 000 strank). Cilj je `odhod`. `id_stranke` ni značilka.
2. Tabela ima 64 značilk: 58 številskih in 6 besedilnih. Stolpcem iz 1. modula so dodani novi
   (3- in 6-mesečna povprečja, razmerja, logaritmi).
3. Skrite oznake (`U`, `UNKW`) in nemogoče negativne vrednosti so postale manjkajoče vrednosti.
4. Delitev 60 / 20 / 20: naključna, z enakim deležem odhodov, seme 42. Vsi imamo iste stranke v istih množicah.
5. Manjkajoče številke so zapolnjene z **mediano učne množice**. Stolpci `..._manjka` povedo, kje je vrednost manjkala.
6. Besedilni stolpci so zapisani tudi kot stolpci 0/1 (na primer `regija_CHICAGO_AREA`).
   Skupaj je 96 stolpcev, ki jih lahko podamo modelu.
7. Osamelcev ne odstranimo in značilk ne skaliramo: drevo vpraša samo, ali je vrednost nad ali pod pragom.
8. V prikazu uporabimo 16 številskih značilk iz 1. modula, da so modeli na lestvici primerljivi.
9. Modele primerjamo na validacijski množici. Testna ostane zaprta do tekmovanja na koncu klasifikacijskega dela.

</details>

In [ ]:
ucni = preberi_tabelo(PODATKI / "train.csv", shema)
validacijski = preberi_tabelo(PODATKI / "val.csv", shema)

ZNACILKE = TELEKOM_STEVILSKE                                           # 16 številskih značilk iz 1. modula
X_ucni, y_ucni = ucni[ZNACILKE], ucni["odhod"]
X_val, y_val = validacijski[ZNACILKE], validacijski["odhod"]
print(f"Učnih strank: {len(ucni)}, validacijskih: {len(validacijski)}, od teh je odšlo {y_val.mean():.1%}")
print(f"Stolpcev v tabeli: {ucni.shape[1]}, manjkajočih vrednosti: {int(ucni.isna().sum().sum())}")

---
## A1. Tri drevesa, trije naključni vzorci
Naredimo to, kar dela gozd, ročno in samo trikrat:
- `sample(frac=1, replace=True)` izbere naključen vzorec vrstic **z vračanjem** (nekatere stranke so v njem večkrat, nekaterih ni),
- `max_features=4` pomeni, da sme drevo pri vsakem vprašanju izbirati samo med 4 naključnimi značilkami od 16.

> **Povezava s 1. modulom:** `sample()` smo že uporabili, ko smo za razsevni diagram vzeli naključen vzorec točk.
> Tu vsako drevo dobi svoj naključni vzorec strank.

In [ ]:
fig, osi = plt.subplots(1, 3, figsize=(19, 4.2))
for os, seme in zip(osi, [1, 5, 10]):                                         # tri različna semena naključja
    vzorec = ucni.sample(frac=1, replace=True, random_state=seme)             # naključen vzorec vrstic
    drevo = DecisionTreeClassifier(max_depth=2, max_features=4, random_state=seme).fit(vzorec[ZNACILKE], vzorec["odhod"])
    narisi_drevo(drevo, ZNACILKE, ax=os, pisava=8)
    os.set_title(f"seme {seme}: validacijska točnost {drevo.score(X_val, y_val):.3f}")
plt.show()

Tri drevesa, tri različna prva vprašanja. Vsako drevo vidi nekoliko drugačen vzorec strank in značilk.
Če njihove napovedi povprečimo, se lahko del nihanja posameznih dreves zmanjša. To je ideja gozda.

## A2. Gozd iz 100 dreves
`RandomForestClassifier` naredi isto, 100-krat. `n_jobs=-1` uporabi vsa jedra računalnika.

In [ ]:
gozd = RandomForestClassifier(n_estimators=100, min_samples_leaf=20, random_state=SEME, n_jobs=-1)
gozd.fit(X_ucni, y_ucni)

drevo_8 = DecisionTreeClassifier(max_depth=8, random_state=SEME).fit(X_ucni, y_ucni)    # najboljše posamezno drevo

pd.DataFrame({"učna": [drevo_8.score(X_ucni, y_ucni), gozd.score(X_ucni, y_ucni)],
              "validacijska": [drevo_8.score(X_val, y_val), gozd.score(X_val, y_val)]},
             index=["eno drevo (globina 8)", "gozd (100 dreves)"]).round(3)

## A3. Posamezna drevesa proti gozdu
Dreves v gozdu ne zavržemo: so v `gozd.estimators_`. Izmerimo točnost vsakega posebej in jo primerjamo s točnostjo gozda.
(Drevesa v gozdu delajo s tabelo brez imen stolpcev, zato jim damo `X_val.values`.)

In [ ]:
tocnosti = pd.DataFrame({"točnost": [(drevo.predict(X_val.values) == y_val.values).mean() for drevo in gozd.estimators_]})
tocnost_gozda = gozd.score(X_val, y_val)
print(f"Posamezna drevesa: od {tocnosti['točnost'].min():.3f} do {tocnosti['točnost'].max():.3f}, "
      f"povprečje {tocnosti['točnost'].mean():.3f}.   Gozd: {tocnost_gozda:.3f}")

(ggplot(tocnosti, aes(x="točnost"))
 + geom_histogram(bins=15, fill=MODRA)
 + geom_vline(xintercept=tocnost_gozda, color=ORANZNA, size=1.5)
 + labs(title="Točnost 100 dreves in točnost gozda",
        subtitle="modro: posamezna drevesa, oranžna črta: gozd", x="validacijska točnost", y="število dreves"))

Nobeno posamezno drevo ni tako dobro kot vsa skupaj.

> **Povezava s 1. modulom:** samo povprečje skrije razpršenost. Zato nismo izpisali le povprečne točnosti dreves,
> ampak smo narisali histogram vseh 100.

## A4. Glasovanje in verjetnost
Za pet strank preštejemo, koliko dreves napove "odide". To je dobra **predstava** o tem, kako gozd odloča.
**Verjetnost**, ki jo vrne `predict_proba`, pa je **povprečje verjetnosti** vseh dreves, ne delež glasov.

In [ ]:
pet_strank = X_val.sample(5, random_state=6)

glasovi = np.array([drevo.predict(pet_strank.values) for drevo in gozd.estimators_])                    # 100 x 5: 0 ali 1
verjetnosti_dreves = np.array([drevo.predict_proba(pet_strank.values)[:, 1] for drevo in gozd.estimators_])

pd.DataFrame({"dreves glasuje 'odide'": glasovi.sum(axis=0).astype(int),
              "povprečje verjetnosti dreves": verjetnosti_dreves.mean(axis=0).round(2),
              "gozd.predict_proba": gozd.predict_proba(pet_strank)[:, 1].round(2),
              "dejansko": y_val.loc[pet_strank.index].values}, index=pet_strank.index)

Drugi in tretji stolpec sta enaka: verjetnost gozda je povprečje verjetnosti dreves.
Za eno stranko narišimo, kaj pravi vsako od 100 dreves:

In [ ]:
ena = pd.DataFrame({"verjetnost": verjetnosti_dreves[:, 1]})          # druga od petih strank

(ggplot(ena, aes(x="verjetnost"))
 + geom_histogram(bins=20, fill=MODRA)
 + geom_vline(xintercept=ena["verjetnost"].mean(), color=ORANZNA, size=1.5)
 + geom_vline(xintercept=0.5, linetype="dashed")
 + labs(title="Ena stranka, 100 dreves",
        subtitle=f"oranžna črta: povprečje = verjetnost gozda ({ena['verjetnost'].mean():.2f}); črtkano: 0,5",
        x="verjetnost odhoda po mnenju posameznega drevesa", y="število dreves"))

## A5. Koliko dreves potrebujemo?

In [ ]:
vrstice = []
for stevilo in [1, 3, 10, 30, 100]:
    g = RandomForestClassifier(n_estimators=stevilo, min_samples_leaf=20, random_state=SEME, n_jobs=-1).fit(X_ucni, y_ucni)
    vrstice.append({"dreves": stevilo, "validacijska točnost": g.score(X_val, y_val)})
po_stevilu = pd.DataFrame(vrstice)
print(po_stevilu.round(3).to_string(index=False))

(ggplot(po_stevilu, aes(x="dreves", y="validacijska točnost"))
 + geom_line(color=MODRA, size=1.2)
 + geom_point(color=MODRA, size=3)
 + scale_x_log10(breaks=[1, 3, 10, 30, 100])
 + labs(title="Več dreves, boljša napoved — do neke meje", x="število dreves (logaritemska os)", y="validacijska točnost"))

Na tej validacijski množici največji prirastek točnosti dobimo s prvimi desetimi drevesi.
Med 30 in 100 se točnost skoraj ne spremeni, učenje pa traja dlje.

## A6. Nova vrstica na lestvici "100 klicev"
Na lestvico postavimo vse modele do zdaj: drevo, logistično regresijo in gozd. Logistično regresijo ponovno naučimo
na istih 16 številskih značilkah, da je primerjava poštena; v sklopu o logistični regresiji je lahko uporabljala še druge stolpce.
Značilke zanjo standardiziramo s povprečjem in odklonom **učne** množice (1. modul).

Gozd naučimo še z dvema drugima semenoma, da vidimo, koliko je lestvica odvisna od naključja.

In [ ]:
verjetnosti_gozd = pd.Series(gozd.predict_proba(X_val)[:, 1], index=X_val.index)
drevo_5 = DecisionTreeClassifier(max_depth=5, random_state=SEME).fit(X_ucni, y_ucni)
verjetnosti_drevo = pd.Series(drevo_5.predict_proba(X_val)[:, 1], index=X_val.index)
skala = StandardScaler().fit(X_ucni)                                   # povprečje in odklon samo iz učne množice
logisticna = LogisticRegression(max_iter=1000).fit(skala.transform(X_ucni), y_ucni)
verjetnosti_logisticna = pd.Series(logisticna.predict_proba(skala.transform(X_val))[:, 1], index=X_val.index)

vrstice = [
    {"izbira": "naključno (pričakovano)", "odšlo od 100": round(100 * y_val.mean()), "odšlo od 1 000": round(1000 * y_val.mean())},
    vrstica_lestvice("drevo (globina 5)", verjetnosti_drevo, y_val),
    vrstica_lestvice("logistična regresija", verjetnosti_logisticna, y_val),
    vrstica_lestvice(f"naključni gozd (100 dreves, seme {SEME})", verjetnosti_gozd, y_val),
]
for seme in [1, 2]:                                   # isti gozd, drugo seme naključja
    g = RandomForestClassifier(n_estimators=100, min_samples_leaf=20, random_state=seme, n_jobs=-1).fit(X_ucni, y_ucni)
    vrstice.append(vrstica_lestvice(f"naključni gozd (100 dreves, seme {seme})",
                                    pd.Series(g.predict_proba(X_val)[:, 1], index=X_val.index), y_val))
pd.DataFrame(vrstice)

Pri **100 izbranih validacijskih strankah** drevo najde 86 dejanskih odhodov, logistična regresija 62, gozd pa 84, 85 ali 86 glede na seme.
Ta razlika je majhna in občutljiva na izbor strank z enakimi ocenami. Pri **1 000** izbranih strankah gozd
v teh treh zagonih najde 776–790 odhodov, drevo 720, logistična regresija 621. Na tej validacijski množici je prednost gozda pri večjem izboru jasnejša.
Zato pogledamo oba obsega, ne le prvih 100.

## A7. Kaj smo izgubili?
Eno drevo smo lahko prebrali kot seznam pravil. Sto dreves ne moremo.
Katere značilke model uporablja največ (**pomembnost značilk**), pride na vrsto pozneje.

---
Naprej: zvezek **`B_nakljucni_gozdovi`**. Tam gozdu izbirate tudi stolpce: tabela ima 96 stolpcev, ki jih lahko podamo modelu.